# GGUF Export — no GPU, no unsloth

Loads the LoRA adapter from HF Hub using plain `transformers` + `peft`,
merges weights, then converts to GGUF using a fresh llama.cpp clone.

**Requirements:** ~16 GB RAM. No GPU needed.

In [ ]:
# Install dependencies (skip if already installed in your conda env)
!pip install --quiet transformers peft accelerate huggingface_hub safetensors sentencepiece protobuf

In [ ]:
HF_USERNAME     = "CopyleftCultivars"
HF_ADAPTER_REPO = f"{HF_USERNAME}/gemma4-natural-farming-lora"
HF_GGUF_REPO    = f"{HF_USERNAME}/gemma4-natural-farming-gguf"
MERGED_DIR      = "/tmp/gemma4-merged"
GGUF_F16        = "/tmp/gemma4.F16.gguf"
GGUF_Q4         = "/tmp/gemma4.Q4_K_M.gguf"
LLAMA_CPP_DIR   = "/tmp/llama.cpp"
print(f"Adapter : {HF_ADAPTER_REPO}")
print(f"GGUF out: {HF_GGUF_REPO}")

In [3]:
from huggingface_hub import login
import getpass

# Paste your HF write token when prompted (won't echo to screen)
token = getpass.getpass("HuggingFace write token: ")
login(token=token)
print("Logged in.")

HuggingFace write token:  ········


Logged in.


In [ ]:
# Clone a fresh llama.cpp to get the latest Gemma 4 support.
import subprocess, os

if not os.path.exists(LLAMA_CPP_DIR):
    subprocess.run(["git", "clone", "--depth=1",
                    "https://github.com/ggerganov/llama.cpp",
                    LLAMA_CPP_DIR], check=True)
else:
    print("Already cloned, pulling latest...")
    subprocess.run(["git", "-C", LLAMA_CPP_DIR, "pull"], check=True)

req = f"{LLAMA_CPP_DIR}/requirements/requirements-convert_hf_to_gguf.txt"
if os.path.exists(req):
    subprocess.run(["pip", "install", "--quiet", "-r", req], check=True)

# Quick check: does this llama.cpp know about Gemma 4?
grep = subprocess.run(["grep", "-rl", "gemma4", f"{LLAMA_CPP_DIR}/"],
                      capture_output=True, text=True)
print("Gemma 4 support:", "FOUND" if grep.stdout.strip() else "NOT FOUND (conversion may fail)")

In [ ]:
# Load base model + LoRA, merge weights, save to disk.
# RAM peak: ~10 GB for a 4B model in bfloat16.
import torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import hf_hub_download

# Read base model name from the adapter's own config
adapter_cfg_path = hf_hub_download(HF_ADAPTER_REPO, "adapter_config.json")
with open(adapter_cfg_path) as f:
    BASE_MODEL = json.load(f)["base_model_name_or_path"]
print(f"Base model: {BASE_MODEL}")

print("Loading base model (bfloat16, CPU — takes a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print("Attaching LoRA adapter...")
model = PeftModel.from_pretrained(model, HF_ADAPTER_REPO)

print("Merging LoRA weights...")
model = model.merge_and_unload()

import os
os.makedirs(MERGED_DIR, exist_ok=True)
print(f"Saving merged model to {MERGED_DIR} ...")
model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("Done.")

In [ ]:
# Convert merged model → F16 GGUF (errors print live, no swallowing)
import subprocess

converter = f"{LLAMA_CPP_DIR}/convert_hf_to_gguf.py"
print("Converting to F16 GGUF...")
result = subprocess.run(["python", converter,
                         "--outfile", GGUF_F16,
                         "--outtype", "f16",
                         MERGED_DIR])
if result.returncode != 0:
    raise RuntimeError("F16 conversion failed — check output above for the real error.")
print(f"F16 GGUF: {GGUF_F16}")

In [ ]:
# Build llama-quantize (one-time, ~2 min) then quantize F16 → Q4_K_M
import subprocess, os

quantize_bin = f"{LLAMA_CPP_DIR}/build/bin/llama-quantize"
if not os.path.exists(quantize_bin):
    print("Building llama-quantize...")
    build_dir = f"{LLAMA_CPP_DIR}/build"
    os.makedirs(build_dir, exist_ok=True)
    subprocess.run(["cmake", "..", "-DCMAKE_BUILD_TYPE=Release",
                    "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_BUILD_EXAMPLES=OFF"],
                   cwd=build_dir, check=True)
    subprocess.run(["cmake", "--build", ".", "--target", "llama-quantize", "-j4"],
                   cwd=build_dir, check=True)

print("Quantizing to Q4_K_M (~2.5 GB)...")
result = subprocess.run([quantize_bin, GGUF_F16, GGUF_Q4, "Q4_K_M"])
if result.returncode != 0:
    raise RuntimeError("Quantization failed.")
print(f"Q4_K_M GGUF: {GGUF_Q4}")

In [ ]:
# Push Q4_K_M GGUF to HuggingFace Hub
from huggingface_hub import HfApi
import os

api = HfApi()
api.create_repo(repo_id=HF_GGUF_REPO, exist_ok=True, private=False)
api.upload_file(
    path_or_fileobj=GGUF_Q4,
    path_in_repo=os.path.basename(GGUF_Q4),
    repo_id=HF_GGUF_REPO,
)
print(f"Done — https://huggingface.co/{HF_GGUF_REPO}")